<a href="https://colab.research.google.com/github/Iberasa3/Vespa-velutina-moves/blob/main/Avispas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd

# El separador '\t' le dice a pandas que use el tabulador
df = pd.read_csv('avispas_avistamientos.csv', sep='\t') #Duh el separador era esta shit

# Ahora sí, vamos a comprobar que se han separado
print(f"Número de columnas: {df.shape[1]}")
print(f"Nombres de las columnas: {df.columns.tolist()[:5]}...")

/tmp/ipykernel_189/551559319.py:4: DtypeWarning: Columns (14,16,17,36,37,38,39,40,41,43,44,45,46,48) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('avispas_avistamientos.csv', sep='\t') #Duh el separador era esta shit


Número de columnas: 50
Nombres de las columnas: ['gbifID', 'datasetKey', 'occurrenceID', 'kingdom', 'phylum']...


Efectivamente este es mi proyecto de avispas, de momento no tengo mucho más que añadir. Cuando termine de limpiar este dataset proyectaré las coordenadas en GEE

Un par de apuntes para mí mismo, probablemente acabemos considerando a todas las subespecies de avispa asiatica como una sola para no tener que liarnos entre las distintas subespecies.

Varios factores que tengo que tener en cuenta para poder filtrar bien los datos que se me están presentando:

1- Los datos anteriores a 2005, si los hay, no hay que tenerlos en cuenta porque Vespa velutina entró a europa en 2004 (¿hecho?)

2- Cuidado con null island (hecho)

3- cuidado con los rangos en los que el avistamiento tiene gran índice de error (hecho)

4- cuidado con los avistamientos no-salvajes (hecho, preservados eliminados)

5- hay que chekiar las diferentes subespecies por si acaso (hecho)

6- hay que chekiar solo los que sean de europa (hecho)


Tengo que leer algunos papers para que esto parezca más académico, pero tampoco pasarme 150 días leyendo papers para justificar toda la literatura


In [4]:
df['infraspecificEpithet'].unique() #Si la celda está vacía entonces asumimos que es la versión común de la vespa

array([nan, 'nigrithorax', 'variana', 'auraria', 'flavitarsus',
       'velutina', 'karnyi', 'celebensis', 'ardens', 'sumbana',
       'divergens', 'floresiana', 'timorensis'], dtype=object)

In [5]:
paises_europa = ['ES', 'FR', 'PT', 'BE', 'IT', 'DE', 'CH', 'LU', 'GB', 'IE', 'NL'] #El análisis se basa en europa así que nos quedamos solo con los países europeos. La mayoría de registros de todas formas son europeos.
df = df[df['countryCode'].isin(paises_europa)]
print(f"Pasamos de {len(df)} a {len(df)} registros.")

Pasamos de 372895 a 372895 registros.


In [6]:
# Contamos cuántos registros hay por país y lo ordenamos de mayor a menor
conteo_paises = df['countryCode'].value_counts()

print("--- Avistamientos por País en Europa ---")
print(conteo_paises)

--- Avistamientos por País en Europa ---
countryCode
BE    201699
NL     72038
FR     54466
CH     20092
DE     17921
ES      3518
PT      2251
LU       642
IT       211
IE        37
GB        20
Name: count, dtype: int64


In [7]:
#Quitamos todas las coordenadas basura y los nulls
df = df[(df['decimalLatitude'] != 0.0) & (df['decimalLongitude'] != 0.0)]
df = df.dropna(subset=['decimalLatitude', 'decimalLongitude'])

print(f"Registros originales: {len(df)}")
print(f"Registros después de haber quitado los nulls y los null-islands: {len(df)}")

Registros originales: 372450
Registros después de haber quitado los nulls y los null-islands: 372450


In [8]:
#Quitamos las instancias en las que la incertidumbre es superior a 1km. Habría que ver por qué coño existen incertidumbres tan grandes
umbral_maximo = 1000
df = df[(df['coordinateUncertaintyInMeters'] <= 1000) & (df['year'] >= 2005)]

Hay que tener cuidado con las instancias que se hayan tomado en el mismo instante, puede ser que un pibe haya apuntado varias veces a una misma colmena o incluso a una misma avispa, y que por ello haya zonas sobrerepresentadas.
Ahora habría que ver cómo chota hacemos esto del spatial thinning.
Como hemos descargado la versión simple del Darwin Core igual tenemos datos que nos ayudan con esto. De hecho igual en un futuro lo hago directamente con DWC

In [9]:
df = df[df['basisOfRecord'] != 'PRESERVED_SPECIMEN']
print(df['basisOfRecord'].unique())

['HUMAN_OBSERVATION']


Vamos con el spatial thinning.

Podemos aplicar una agrupación por celda, útil para evitar sobremuestreos pero falla para detectar zonas más idóneas en las que las avispas hayan podido construir sus colmenas. Luego podemos sumarlo a un check de correlación para ver si nos ha salido bien.

La cosa es que si vamos a aplicar un random forest se asume que cada dato va a ser independiente, el modelo no va a saber que los distintos puntos que están cerca entre sí pueden reflejar una colmena próxima. Además al reflejar los puntos en el propio mapa la resolución no va a dar para más, así que creo que hacer el spatial thinning sin tener en cuenta la posibilidad de colmena es, de momento, suficiente. Esto es un modelo de distribución, no uno de abundancia.

¿Cuánto vuela una avispa desde su nido?
Por lo que he leído en internet de fuentes fiables, todas coinciden en que rara vez sobrepasan 1km o el kilometro y medio desde sus nidos, así que si hay algún avistamiento de avispa debe de haber un nido relativamente cerca casi seguro



In [10]:
num_duplicados = df.duplicated().sum()
print(f"Registros con el mismo gbifID: {num_duplicados}")
df.drop_duplicates(subset='gbifID', inplace=True)

Registros con el mismo gbifID: 0


Hemos comprobado también que no haya avistamientos con un ID repetido. Se asume que los estándares de Darwin Core no permiten duplicados en sus datasets, pero por si acaso

In [11]:
# 0.015 grados de latitud son aproximadamente 1.1 km, así que cada celda va a tener aprox 1,1km x 1,1km
# Es una resolución estándar para modelos climáticos de 1km.
res = 0.015

# Dividimos por la resolución, redondeamos al entero más cercano y volvemos a multiplicar.
df['lat_grid'] = (df['decimalLatitude'] / res).round() * res
df['lon_grid'] = (df['decimalLongitude'] / res).round() * res

print(f"originalmente hay {len(df)} registros")
df = df.drop_duplicates(subset=['lat_grid', 'lon_grid'], keep='first').copy()

print(f"Dataset tras thinning (1km): {len(df)} registros")

df[['decimalLatitude', 'lat_grid', 'decimalLongitude', 'lon_grid']].head()

originalmente hay 231961 registros
Dataset tras thinning (1km): 29307 registros


,decimalLatitude,lat_grid,decimalLongitude,lon_grid
2,51.11106,51.105,4.07784,4.080
3,51.21826,51.225,2.90049,2.895
4,50.86031,50.865,4.62720,4.620
5,51.03947,51.045,3.74158,3.735
6,50.95183,50.955,3.68754,3.690


Nos hemos quitado casi el 90% del dataset eliminando avistamientos extremadamente cercanos. Ahora tenemos 29.300 registros para entrenar nuestro modelo

Eso sí, lo mejor tratar de forma distinta los datos que tengan poca fiabilidad y los datos que sean muy exactos en su radio de avistamiento?... Lo que ocurre es que las avispas vuelan, y pueden volar ocasionalmente a zonas que no le son adecuadas para nidificar.


In [12]:
# Contamos cuántos registros hay por país y lo ordenamos de mayor a menor, estos son las casillas donde ha habido avistamiento
conteo_paises = df['countryCode'].value_counts()
conteo_total = df
print("--- Avistamientos por País en Europa ---")
print(conteo_paises)

--- Avistamientos por País en Europa ---
countryCode
BE    10519
FR     8792
DE     7482
ES     1247
PT      760
LU      184
NL      124
CH       94
IT       91
IE        9
GB        5
Name: count, dtype: int64


In [13]:
import sys
import geemap
import ee # Import the Earth Engine library

# Install geojson library if not already installed
!pip install geojson

# Authenticate and Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project='vespa-489513')

columnas_necesarias = ['decimalLatitude', 'decimalLongitude', 'year']
df = df[columnas_necesarias].copy()

df = df.dropna(subset=['decimalLatitude', 'decimalLongitude']) #Eliminamos los valores na

puntos_geospatiales = geemap.pandas_to_ee( #Estos sin mis puntos según la latitud y longitud a partir de ahora
    df,
    latitude='decimalLatitude',
    longitude='decimalLongitude'
)


Map = geemap.Map()
Map.addLayer(puntos_geospatiales, {'color': 'red'}, 'Vespa velutina (Optimized)')
Map.centerObject(puntos_geospatiales, 6)
Map


"\nMap = geemap.Map()\nMap.addLayer(puntos_geospatiales, {'color': 'red'}, 'Vespa velutina (Optimized)')\nMap.centerObject(puntos_geospatiales, 6)\nMap\n"

Ahora toca hacer que un modelo randomForest me aprenda las características de los sitios donde puede haber más avispas Vespa Velutina.

Hasta aquí hemos:
1- Conseguido el dataset desde GBIF, que registra observaciones de la Vespa velutina en un formato Darwin core (aunque de momento con la versión procesada nos sirve)

2- Hemos filtrado el dataset (1) quedándonos solo con los países de europa (las condiciones de Asía no nos sirven si son parecidas?¿?), (2) filtrando y eliminando las coordenadas no-útiles, (3) eliminando las instancias cuya tasa de error es superior a 1km, (4) quedándonos solo con las especies vistas en estado silvestre, (5) quedándonos solo con los avistamientos realizados a partir de 2005, y (6), eliminando las posibles observaciones duplicadas.

3- Hemos hecho una refracción espacial de 1,6km para que ciertos avistamientos no se vean representados, convirtiendo el mapa en casillas cuyos valores son o 1 (en cuyo caso la avispa está presente) o 0, en cuyo caso no lo está

4- Hemos reflejado dichas casillas en el mapa con GEE.

El siguiente paso es aprender los valores geográficos de las casillas en las que SÍ hay avispa y de las casillas en las que NO las hay. Luego aplicaremos un modelo random forest para poder predecir futuras expansiones. La casilla 0 estará representada por pseudoausencias, que enseñarán al modelo en qué condiciones NO hay avispas.

PROBLEMA: ¿CÓMO SABEMOS SI NO HAY AVISPAS PORQUE TODAVÍA NO HAN LLEGADO O PORQUE LAS CONDICIONES NO SON FAVORABLES?

Plus sería buena idea descargarme el dataset limpio para no tener que andar limpiándolo cada vez

Usaremos WorldClim para las variables bioclimáticas, NASA SRTM para valores de presión y altura, y MODIS/WorldCover para saber el tipo de cobertura terrestre sobre la que estamos trabajando.

¿Es esto útil u óptimo tras haber simplifcado cada punto a una casilla? en 1km puede cambiar mucho la altura y la distribución de suelo...
y esto de arriba no tiene mucho sentido porque con el thinning nos quedamos siempre con un punto.

In [14]:
# --- CONFIGURACIÓN DE PREDICTORES ---

# Definimos el área de interés para no petar el maps
# Creamos un bounding box que envuelva todos los avistamientos con un margen de 50km por si acaso
aoi = puntos_geospatiales.geometry().bounds().buffer(50000) #aoi devuelve una figura geométrica

# Cargamos las variables relacionadas al clima de la región
# BIO01=Temp Media, BIO12=Precipitación Anual, BIO05/06=Máximas y mínimas
bioclim = ee.Image('WORLDCLIM/V1/BIO').clip(aoi) #bioclim es una imagen multicanal (19 bandas) en mi rango de interés aoi

# Cargamos las variables relacionadas a la topología
# La elevación influye MUCHO en dónde la avispa decide poner el nido (prefieren valles antes que montañas)
elevacion = ee.Image('USGS/SRTMGL1_003').clip(aoi)  #elevacion es una imagen monocanal (solo altitud) en mi rango de interes aoi

# Cargamos el tipo de cobertura del suelo.
# Este mapa clasifica el suelo: 10=Bosque, 50=Urbano, 60=Vegetación rala/Playas...
# Usamos.first() porque es una colección de imágenes que cubren el globo
terreno = ee.ImageCollection("ESA/WorldCover/v100").first().clip(aoi).select(['Map'], ['landcover']) #terreno es otra imagen monocanal (solo tipo de suelo) en mi rango de interés aoi
#Terreno tiene muy muy buena resolución (10x10) y nos dirá qué tipo de suelo es el que hay en cada punto
#Realmente no le veo mucho sentido porque cada casilla es de (1kmx1km) y el tipo de terreno puede no verse bien representado


# Fusionamos los mapas en una sola imagen de múltiples bandas y poder darle a la IA la información sobre cada casilla
predictores = bioclim.addBands(elevacion).addBands(terreno).select([
    'bio01', 'bio05', 'bio06', 'bio12', 'elevation', 'landcover' #Los preeidc
])

# --- SAMPLING (MUESTREO) ---

# scale=1000: La IA mirará qué hay en el kilómetro cuadrado alrededor del punto
datos_presencia_mureados = predictores.sampleRegions(
    collection=puntos_geospatiales,
    properties=['year'], # Guardamos el año para ver la evolución histórica de la invasión en un futuro (igual no lo aplicamos de momento)
    scale=1000,
    geometries=True # Adjunta las coordenadas exactas de cada píxel a la tabla resultante, esto nos es útil porque luego podemos querer volver a representarlas
)

Para ver cómo ha quedado la variable datos_presencia_mureados:

bio01: **temepratura media** en celsius x 10

bio05: **temperatura máxima** en celsius x 10

bio06: **temperatura mínima** en celsius x 10

bio12: precipitaciones anuales acumuladas en mm

elevation: metros sobre el nivel del mar

landcover: tipo de terreno sobre el que se encuentra cada avistamiento

year: año del avistamiento

In [15]:

print(bioclim.bandNames().getInfo())
print("HEAD DEL DF_MUESTREADO ABAJO")
df_muestreado = geemap.ee_to_df(datos_presencia_mureados)
print(df_muestreado.head())


'\nprint(bioclim.bandNames().getInfo())\nprint("HEAD DEL DF_MUESTREADO ABAJO")\ndf_muestreado = geemap.ee_to_df(datos_presencia_mureados)\nprint(df_muestreado.head())\n'

In [16]:

#Asignamos colores a cada tipo de terreno
landcover_palette = {
    10: '006400', # Bosques (Verde oscuro)
    20: 'ffbb22', # Matorral (Naranja)
    30: 'ffff4c', # Pastizales (Amarillo)
    40: 'f096ff', # Cultivos (Rosa)
    50: 'fa0000', # Urbano / Asfalto (Rojo)
    60: 'b4b4b4', # Vegetación rala / Playas (Gris)
    80: '0064ff', # Agua (Azul)
    90: '00bb88', # Humedales (Verde azulado)
    100: 'ffff4c' # Musgo / Liquen
}


def aplicar_estilo(feature):
    lc_value = ee.Number(feature.get('landcover'))
    keys = ee.List(list(landcover_palette.keys()))
    colors = ee.List(list(landcover_palette.values()))
    index = keys.indexOf(lc_value)
    color = ee.Algorithms.If(index.gte(0), colors.get(index), 'ffffff')

    return feature.set('style', {
        'color': color,
        'pointSize': 3,
        'pointShape': 'circle',
        'width': 1
    })

puntos_estilados = datos_presencia_mureados.map(aplicar_estilo)


Map = geemap.Map()
Map.add_layer(ee.ImageCollection("ESA/WorldCover/v100").first(), {}, 'Mapa Terreno Global', False)
Map.addLayer(puntos_estilados.style(styleProperty='style'), {}, 'Vespa por tipo de Terreno')
Map.centerObject(datos_presencia_mureados, 6)
Map

"""

'\n#Asignamos colores a cada tipo de terreno\nlandcover_palette = {\n    10: \'006400\', # Bosques (Verde oscuro)\n    20: \'ffbb22\', # Matorral (Naranja)\n    30: \'ffff4c\', # Pastizales (Amarillo)\n    40: \'f096ff\', # Cultivos (Rosa)\n    50: \'fa0000\', # Urbano / Asfalto (Rojo)\n    60: \'b4b4b4\', # Vegetación rala / Playas (Gris)\n    80: \'0064ff\', # Agua (Azul)\n    90: \'00bb88\', # Humedales (Verde azulado)\n    100: \'ffff4c\' # Musgo / Liquen\n}\n\n\ndef aplicar_estilo(feature):\n    lc_value = ee.Number(feature.get(\'landcover\'))\n    keys = ee.List(list(landcover_palette.keys()))\n    colors = ee.List(list(landcover_palette.values()))\n    index = keys.indexOf(lc_value)\n    color = ee.Algorithms.If(index.gte(0), colors.get(index), \'ffffff\')\n\n    return feature.set(\'style\', {\n        \'color\': color,\n        \'pointSize\': 3,\n        \'pointShape\': \'circle\',\n        \'width\': 1\n    })\n\npuntos_estilados = datos_presencia_mureados.map(aplicar_estilo)

Esta sección es para enseñarle al modelo dónde NO hay avispas, para ello generamos pseudoausencias. En un principio tenía pensado hacerlas random, pero rápidamente surjen varios problemas:

1- qué pasa si en la zona de avistamiento hay un punto de ausencia cuando en realidad debería de ser de presencia? A lo mejor ese lugar es un bosque profundo y no pasan personas para avistar avispas. O igual el punto random cae en un sitio en el que sí ha habido avistamientos. Hacerlo random, sobre todo teniendo en cuenta que lo hacemos en un rango aoi donde sí se han producido avistamientos, no es una buena idea

2- Si efectivamente el punto random cae en un sitio donde efectivamente la avispa está ausente, ¿cómo sabemos que está ausente porque las condiciones no son las adecuadas? a lo mejor simplemente la avispa todavía no ha llegado a esa zona, o hay una barrera geográfica que le impide llegar aunque las condiciones sí sean las adecuadas en la zona en sí (imaginemos el paraíso de las avispas rodeado de unas montañas inexpugnables, la avispa no llegará al paraíso pero no porque el paraiso no sea bueno, si no porque no tiene forma de llegar ahí).

Haremos un híbrido y luego lo justificaremos en el documento y en el vídeo.

## SM1

Estrategia para generar pseudoausencias 1: SM1 random
Generamos puntos random en nuestra zona de interés que no coincidan con los puntos de presencia. Una vez generados los visualizamos y guardamos en un dataset para poder compararlos en un futuro y no tener que ejecutar esta sección cada vez

In [17]:

#GENERACIÓN DE AUSENCIAS SM1

#Creamos la imagen de los puntos de presencia
presencia_img = ee.Image().paint(puntos_geospatiales, 1000)

# Creamos una imagen de valor 1 en toda la AOI y "borramos" donde hay presencias
# Usamos un buffer de 1000m (1km) para asegurar que no caigan en el mismo píxel
mascara_exclusion = presencia_img.fastDistanceTransform().sqrt().gt(1)
area_muestreo = ee.Image(1).clip(aoi).updateMask(mascara_exclusion) #Area muestreo es el área permitida donde puede haber ausencias

# Generamos las 29.306 pseudo-ausencias (Seguimos un ratio 1:1), aunque se pueden generar algunos menos
# si el punto random cae justo en una presencia y se descarta.

pseudo_ausencias = area_muestreo.sample(
    region=aoi,
    scale=1000,
    numPixels=29306,
    seed=67, #Six-seven
    geometries=True
)

# Etiquetamos y unimos el dataset
# Clase 1 = Presencia, Clase 0 = Pseudo-ausencia
presencias = puntos_geospatiales.map(lambda f: f.set('class', 1))
ausencias = pseudo_ausencias.map(lambda f: f.set('class', 0))

#El dataset final
dataset_total = presencias.merge(ausencias)
print(f"Dataset SM1 tiene: {dataset_total.size().getInfo()} puntos totales. Deberían de ser cerca de 58.612")

Dataset SM1 tiene: 58153 puntos totales. Deberían de ser cerca de 58.612


In [18]:

# SECCIÓN DE VISUALIZACIÓN SM1
Map = geemap.Map()
Map.add_basemap('HYBRID')

estilo_vespa = {'color': 'red', 'pointSize': 4, 'pointShape': 'circle', 'width': 1}
estilo_vacio = {'color': '00ffff', 'pointSize': 4, 'pointShape': 'square', 'width': 1} # Cian más grande

# Metemos una opción de máscara para que se vea en qué región se pueden colocar los puntos
vis_mask = {
    'min': 0,
    'max': 1,
    'palette': ['000000', 'yellow'], # Changed 'transparent' to '000000' (black)
    'opacity': 0.35 # Menos opaco para ver el terreno
}
Map.addLayer(area_muestreo, vis_mask, 'Debug: Área de Muestreo (Permitida)')
Map.addLayer(ausencias.style(**estilo_vacio), {}, 'Pseudo-ausencias (Clase 0)')
Map.addLayer(presencias.style(**estilo_vespa), {}, 'Presencias Vespa (Clase 1)')

Map.centerObject(puntos_geospatiales, 6)
Map


KeyboardInterrupt: 

Exportamos las pseudoausencias generadas con SM1 para no tener que hacer el proceso de limpieza cada vez y luego poder comparar los modelos utilizando diferentes técnicas.
Esto lo mantengo comentado por si acaso tengo que volver a hacerlo.



In [19]:
"""
# EXTRACCIÓN Y GUARDADO DE COORDENADAS

def extraer_posicion(feature):
    coords = feature.geometry().coordinates()
    return feature.set({
        'latitude': coords.get(1),
        'longitude': coords.get(0)
    })


ausencias_finales = ausencias.map(extraer_posicion).select(['latitude', 'longitude', 'class'])
nombre_csv = 'SM1_absences_COORDENADAS.csv'
geemap.ee_to_csv(ausencias_finales, filename=nombre_csv)
"""

🚀 ¡Ahora sí! El archivo 'SM1_absences_COORDENADAS.csv' debería aparecer en segundos en la carpeta de la izquierda.


## REVISIONES

Cosas a arreglar o revisar:

1- el encasillamiento puede no ser óptimo, además podemos estar sobrerrepresentando zonas de clima templado atlántico porque es justo dónde más se ha preocupado la gente de registrar los avistamientos. Pero eso no quiere decir que no se puedan adaptar a otros sitios, ¿o sí?

2- las pseudoausencias, creo, deberían de ser tratadas de otra forma. SEGURO QUE DEBEN DE SER TRATADAS DE OTRA FORMA

3- podemos tratar de forma diferente los datos precisos de los no precisos? No es necesario, ya hemos filtrado los datos no precisos. Además un avistamiento puede darse a varios cientos de metros o más de una colmena.

4- hay que investigar con papers cómo se ha llevado esto a cabo por parte de otras personas. Esto lo haré a la tarde

4.2- MEJORAR EL FEATURE ENGINEERING, CALCULANDO MÁS INFO IMPORTANTE (DISTANCIA A RIOS, A CIUDADES, ILUMINACIÓN...)

5- ENTRENAR EL MODELO, LUEGO HACER VALIDACIÓN CRUZADA (SE VERÁ COMO SE HACE) PARA VER CÓMO DE BUENO ES.

6- REVISAR, DOCUMENTAR, README DE GITHUB Y PRODUCCIÓN DEL VÍDEO


Enlaces útiles:

https://ramirodcrego.com/teaching/gee/

https://developers.google.com/earth-engine/tutorials/community/species-distribution-modeling?hl=es-419

https://www.tandfonline.com/doi/full/10.1080/10095020.2025.2546507#abstract

https://www.researchgate.net/publication/284246225_A_multi-scale_approach_to_identify_invasion_drivers_and_invaders'_future_dynamics

https://onlinelibrary.wiley.com/doi/full/10.1002/ece3.70029

https://besjournals.onlinelibrary.wiley.com/doi/10.1111/j.2041-210X.2011.00172.x

https://biogeography-usc.org/positive/Prediction.html

https://www.sciencedirect.com/science/article/abs/pii/S0006320711001315

https://revistaecosistemas.net/index.php/ecosistemas/article/view/2987/1962

https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0071218 (PAPER SM4)

https://gidahatari.com/ih-es/bioclim-un-sistema-de-analisis-y-prediccion-de-bioclimas (BIOCLIM)

https://www.researchgate.net/publication/226562656_DOMAIN_a_flexible_modelling_procedure_for_mapping_potential_distributions_of_plants_and_animals (DOMAIN)
